# NLP05 — KoChatGPT 업그레이드
KoGPT2 기반 RLHF 3단계 파이프라인: SFT → RM → PPO

**루브릭**
- ① baseline vs SFT 비교
- ② SFT vs RM/PPO 비교
- ③ Decoding 실험 + LLM-as-a-Judge

In [ ]:
import os
os.chdir('/workspace/test/NLP/NLP05')
import sys
sys.path.insert(0, '/workspace/test/NLP/NLP05')
print('Working dir:', os.getcwd())

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers', 'torch', 'nltk',
                'rouge-score', 'matplotlib', '--break-system-packages'])
import nltk
nltk.download('punkt', quiet=True)
print('설치 완료')

## STEP 1 — SFT (지도 미세조정)

### 학습 결과

| 항목 | 값 |
|------|-----|
| Best epoch | 3 |
| 소요 시간 | 239초 |
| cfg.epochs | 3 |
| cfg.batch_size | 8 |
| cfg.lr | 2e-05 |
| cfg.max_len | 128 |

**Epoch별 loss:**

| Epoch | train_loss | val_loss |
|-------|------------|----------|
| 1 | 2.1709 | - |
| 2 | 1.2735 | - |
| 3 | 1.1561 | - |


## STEP 2 — RM (보상 모델 학습)

### 학습 결과

| 항목 | 값 |
|------|-----|
| Best epoch | 3 |
| 소요 시간 | 278초 |
| cfg.epochs | 3 |
| cfg.batch_size | 8 |
| cfg.lr | 1e-05 |
| cfg.max_len | 128 |

**Epoch별 loss:**

| Epoch | train_loss | val_loss |
|-------|------------|----------|
| 1 | 0.7117 | - |
| 2 | 0.6906 | - |
| 3 | 0.6537 | - |


## STEP 3 — PPO (강화학습)

### 학습 결과

| 항목 | 값 |
|------|-----|
| Best epoch | 1 |
| 소요 시간 | 1106초 |
| cfg.epochs | 1 |
| cfg.batch_size | 4 |
| cfg.lr | 1e-05 |
| cfg.kl_coef | 0.1 |
| cfg.clip_eps | 0.2 |

**Epoch별 loss:**

| Epoch | train_loss | val_loss |
|-------|------------|----------|
| 1 | 509.5552 | - |


## STEP 4 — 모델 비교 평가

### ① baseline vs SFT 비교

| 프롬프트 | baseline | SFT |
|----------|----------|-----|
| 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡... | 미샤엘이 만든 화장품 브랜드인 '에스티로더'다. 에스티로더는 미샤엘이 만든 화장품 브랜드다. 에스티로더가 만... | '저는 인공지능 언어모델로써 답변을 생성하고 있으며, 현재 시점에서는 존재하지 않습니다. 따라서 해당 질문에... |
| 컴터 부팅하는데 시간이 오래걸려 | 요. 그런데 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게 이게... | 요?'컴퓨터 부팅은 일반적으로 몇 분에서 몇 시간까지 걸립니다. 그러나 이는 컴퓨터 부팅에 대한 일반적인 규... |
| 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다... | " "그렇습니다." "그렇습니다. 그 대학은 그 대학보다 훨씬 더 규모가 큰 대학입니다." "그렇다면 그 대... | '저는 AI 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 인터넷 검색 결과, 도나 게일 ... |
| 서너 대 밀려있어도 그런데 제가 백번 밀려있길래그 내 ... | 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그래서 그거야. 그... | '그렇다면 그 사람이 다시 버스정류장에 가서 그 사람을 만나서 이야기를 나누고 그 사람이 내일 다시 버스정류... |
| 드라마 <아이리스>의 최고 시청률은 어떠했는가? | #20180720 #미롱_식단   | '<아이리스>의 최고 시청률은 드라마 <아이리스>의 평균 시청률인 약 12%입니다. |


### ② SFT vs PPO 비교

| 프롬프트 | SFT | PPO |
|----------|-----|-----|
| 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡... | '저는 인공지능 언어모델로써 답변을 생성하고 있으며, 현재 시점에서는 존재하지 않습니다. ... | ''''''''''''''''''''''''''''''''''''''''''''''''''... |
| 컴터 부팅하는데 시간이 오래걸려 | 요?'컴퓨터 부팅은 일반적으로 몇 분에서 몇 시간까지 걸립니다. 그러나 이는 컴퓨터 부팅에... | ''''''''''''''''''''''''''''''''''''''''''''''''''... |
| 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다... | '저는 AI 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 인터넷 검색 ... | ''''''''''''''''''''''''''''''''''''''''''''''''''... |
| 서너 대 밀려있어도 그런데 제가 백번 밀려있길래그 내 ... | '그렇다면 그 사람이 다시 버스정류장에 가서 그 사람을 만나서 이야기를 나누고 그 사람이 ... | ''제가 그 차를 놓쳐버렸습니다. 제가 그 차를 놓쳤습니다. 제가 그 차를 놓쳤습니다. 제... |
| 드라마 <아이리스>의 최고 시청률은 어떠했는가? | '<아이리스>의 최고 시청률은 드라마 <아이리스>의 평균 시청률인 약 12%입니다. | ''''''''''''''''''''''''''''''''''''''''''''''''''... |


## STEP 5 — 정량 평가 (BLEU + ROUGE-L)

| 모델 | BLEU | ROUGE-L |
|------|------|---------|  
| baseline | 0.0184 | 0.0000 |
| SFT | 0.1742 | 0.0813 |
| PPO | 0.0736 | 0.0200 |

In [ ]:
import json, matplotlib.pyplot as plt
import config as cfg

with open(f'{cfg.RESULTS_DIR}/quant_summary.json') as f:
    quant = json.load(f)

stages = list(quant.keys())
bleu   = [quant[s].get('avg_bleu', quant[s].get('bleu', 0)) for s in stages]
rouge  = [quant[s].get('avg_rouge_l', quant[s].get('rouge_l', 0)) for s in stages]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(stages, bleu,  color=['#aec6cf','#779ecb','#4a7ebf'])
axes[0].set_title('BLEU Score')
axes[0].set_ylabel('BLEU')
axes[1].bar(stages, rouge, color=['#aec6cf','#779ecb','#4a7ebf'])
axes[1].set_title('ROUGE-L Score')
axes[1].set_ylabel('ROUGE-L')
plt.suptitle('baseline vs SFT vs PPO')
plt.tight_layout()
plt.savefig(f'{cfg.RESULTS_DIR}/quantitative_comparison.png', dpi=150)
plt.show()

## Decoding 실험 (PPO 모델)

### 샘플 1: 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡게 된 곳은?

| 전략 | 출력 |
|------|------|
| greedy | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| beam_2 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| beam_4 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| topk_10 | ''''''''''''''''"''''''''''''''''''''''''''''''''''''''''''''''' |
| topk_50 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| topp_09 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |


### 샘플 2: 컴터 부팅하는데 시간이 오래걸려

| 전략 | 출력 |
|------|------|
| greedy | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| beam_2 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| beam_4 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| topk_10 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| topk_50 | '''''''''"''''''''''''"''''''''''''''''''''''''''''''''''''''''' |
| topp_09 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |


### 샘플 3: 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다니던 대학교는?

| 전략 | 출력 |
|------|------|
| greedy | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| beam_2 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| beam_4 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |
| topk_10 | '''가'''라'란'''"'라'를)라는 '아'로)라고)라는 '아'로)'라는)라는 |
| topk_50 | '''''"''"''에서 ''''''''''''''가'''이'''''가''''''이'''''''' '가' '으로'라고 '성 |
| topp_09 | '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''' |


## LLM-as-a-Judge (SFT vs PPO)

### 샘플 1

**[질문]** 막스와 미샤엘이 설립한 공장 중에 이후 모든 제작을 맡게 된 곳은?

**[응답 A - SFT]** '저는 인공지능 언어모델로써 답변을 생성하고 있으며, 현재 시점에서는 존재하지 않습니다. 따라서 해당 질문에 대한 정확한 답변을 제공할 수 없습니다.

**[응답 B - PPO]** ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

**평가 기준:** 유창성 / 관련성 / 정보성 (각 1~5점)

### 샘플 2

**[질문]** 컴터 부팅하는데 시간이 오래걸려

**[응답 A - SFT]** 요?'컴퓨터 부팅은 일반적으로 몇 분에서 몇 시간까지 걸립니다. 그러나 이는 컴퓨터 부팅에 대한 일반적인 규칙과 절차에 따라 달라질 수 있습니다. 일반적으로 컴퓨터 부팅 시간은 일반적으로 몇 분에서 몇 시간까지 소요됩니다.

**[응답 B - PPO]** ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

**평가 기준:** 유창성 / 관련성 / 정보성 (각 1~5점)

### 샘플 3

**[질문]** 3월 12일 사라진 19살 여대생 도나 게일 맨슨이 다니던 대학교는?

**[응답 A - SFT]** '저는 AI 어시스턴트이기 때문에 정확한 답변을 제공할 수 없습니다. 하지만 인터넷 검색 결과, 도나 게일 맨슨이 다니던 대학교는 미국 캘리포니아주 샌프란시스코에 위치한 캘리포니아 주립대학교입니다.

**[응답 B - PPO]** ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

**평가 기준:** 유창성 / 관련성 / 정보성 (각 1~5점)

## 결론 및 고찰

| 단계 | Best Loss | BLEU | ROUGE-L |
|------|-----------|------|---------|  
| SFT | 1.1561 | 0.1742 | 0.0813 |
| RM | 0.6537 | - | - |
| PPO | 509.5552 | 0.0736 | 0.0200 |

### RLHF 3단계 파이프라인 체험 요약

| 단계 | 핵심 학습 |
|------|----------|
| SFT | 형식과 말투를 학습시킬 수 있다 |
| RM | "좋은 답"을 점수로 정의할 수 있다 |
| PPO | RM이 허술하면 Reward Hacking이 발생한다 |

### 개선 방향
1. `tokenizer.padding_side = 'left'` 적용
2. KL penalty 계수 상향 (0.1 → 0.3~0.5)
3. RM 학습 데이터 품질 개선
4. PPO epochs 증가 또는 배치 크기 조정